# 16 — Strict ten-session RET and corrected ranker

267 prior-close features, ten completed sessions and nine internal overnight gaps. No current trade-day input. Both learners retain their established parameters, 100 models per quarter, 12/3-month WFO and 30-round validation patience. No tuning or arbitrary round/RAM/runtime ceilings.

Eligibility uses historical price/activity/source quality. Flat-opening and zero-gap states retain native NaNs. This replaces the old cohort and its cohort-conditioned labels/payoffs as well as the feature representation; comparisons are not an isolated feature-only experiment. RET uses random stock-row validation; the corrected ranker uses random whole-date validation with fixed complete-outcome dates and equal N2/N3 gross-return stopping.

This notebook only inspects prepared/completed artifacts. It cannot start fits or build an absent release. Licensed outputs stay private.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "scripts").exists():
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR / "scripts"))
from train_session_window import NAME, REPORT, report
RUN = PROJECT_DIR / "data/models" / NAME
if not (RUN / "PINNED.json").exists():
    raise FileNotFoundError("Prepare the authorized run separately; inspection cannot build or train.")
panel = json.loads((RUN / "PANEL.json").read_text())
display(pd.Series({k:v for k,v in panel.items() if k not in ("model_features", "output_sha256")}, name="Preparation"))
display(pd.DataFrame(json.loads((RUN / "PRETRAIN_AUDIT.json").read_text())["checks"].items(), columns=["Check", "Passed"]))
status = json.loads((RUN / "RUN_STATUS.json").read_text()) if (RUN / "RUN_STATUS.json").exists() else {"status":"prepared; zero fits"}
display(status)

## Results on completion

Costs: gross plus 5/10/20 bp per executed entry and exit dollar notional; 10 bp/side is the reporting standard. Compare both N values on the same observed dates across all eight portfolios. Preserve cash and disclose unresolved selected exits and omitted dates. Return EV/R² is defined for calibrated RET forecasts only, not ranker percentiles. Gain is training loss reduction; descriptor/lag summaries show where it is allocated.

In [ ]:
COMPLETE = all((RUN / learner / "RESULTS.json").exists() for learner in ("ret", "ranker"))
if COMPLETE:
    REPORT_DIR = report(PROJECT_DIR)
    comparison = pd.read_csv(REPORT_DIR / "portfolio_comparison.csv")
    display(comparison.query("scope == 'eight_way_common_observed_dates'")[["model","n_per_side","cost_bps_per_side","sessions","annual_compound","sharpe","max_drawdown","omitted_dates"]])
    display(Image(filename=str(REPORT_DIR / "portfolio_comparison.png")))
else:
    print("Training is incomplete. No reduced-member ensemble or partial performance is presented.")

In [ ]:
if COMPLETE:
    for filename in ("ret_feature_importance.png", "ranker_feature_importance.png", "grouped_importance.png", "explained_variance.png"):
        display(Image(filename=str(REPORT_DIR / filename)))
    display(pd.read_csv(REPORT_DIR / "matched_return_EV.csv"))
    display(json.loads((REPORT_DIR / "SUMMARY.json").read_text())["qualifications"])